# Beta-t-EGARCH likelihood with filtered quantities

This companion helper uses the same likelihood and state recursion as `Beta_univ_t_Egarch_Logl`, but additionally returns the filtered quantities needed for interpretation and diagnostics after estimation.


## 0. When to use this function

Call this function after estimation, using the fitted vector `x_s`. Its inputs have the same meanings as in the objective function, ensuring that fitted states are produced under precisely the specification that was optimized.


## 1. Returned objects

The function returns the negative log-likelihood `Logl`, the log-scale path `lam`, standardized residuals `res`, score innovations `u`, and the Beta-type transformation `Beta`. These series underpin the fitted-scale, residual-density, autocorrelation, and PIT diagnostics in the main notebook.

The legacy `fit` output is retained as `y - res` for compatibility with the original workflow.


## 2. Filtering logic

At each date, the current state determines the conditional scale; the standardized residual generates a Student's *t* score; and that score updates the next state. The code therefore produces one-step-ahead filtered quantities in chronological order.

## 3. Function implementation


In [ ]:
function [Logl,lam,res,u,Beta,fit]= Beta_univ_t_Egarch_Logl_fact (par,y,inf,I)

if I==1
    
    mu = par(1); %mean
    omega = par(2); %unconstrained mean lambda scale
    kapa= par(3); %dinamic cond score par
    vega_l = par(4); %log shape parameter
    
    vega=exp(vega_l);

    if size(par,1)==5
        kapa_2=par(5);
    end
    
else
    
    mu = par(1); %mean
    omega = par(2); %unconstrained mean lambda scale
    phi = par(3); %dinamic AR parameter scale
    kapa= par(4); %dinamic cond score par
    vega_l = par(5); %log shape parameter

    vega=exp(vega_l);

    if size(par,1)==6
        kapa_2=par(6);
    end
    
end


T=size(y,1);

lam=zeros(T,1);
res=zeros(T,1);
u=zeros(T,1);
Beta=zeros(T,1);

if inf==1
    Inf=2*vega/(vega+3);
else
    Inf=1;
end

lam_=omega;

for i=1:T
   
	lam(i)=lam_;
	res(i)=(y(i)-mu)*exp(-lam(i));

    Beta(i)=((res(i)^2)/vega)/(1+((res(i)^2)/vega));
    u(i)=((vega+1)*Beta(i)-1)/Inf;
   	
    if I==1
        
        if size(par,1)==5
            lam_=lam(i)+kapa*u(i)+kapa_2*sign(mu-y(i))*(u(i)+1);
        else
            lam_=lam(i)+kapa*u(i);
        end
   	
    else
        
        if size(par,1)==6
            lam_=omega*(1-phi)+phi*lam(i)+kapa*u(i)+kapa_2*sign(mu-y(i))*(u(i)+1);
        else
            lam_=omega*(1-phi)+phi*lam(i)+kapa*u(i);
        end

    end

end

fit=y-res;

LoglresSum=log(ones(T,1)+1/vega*res.^2);

Logl=T*log(gamma((vega+1)/2)/(gamma(vega/2)*sqrt(pi*vega)))-sum(lam)-(vega+1)/2*sum(LoglresSum);

Logl=-Logl;
